# 🚲 Previsão de Demanda — Passo a passo guiado

Este notebook percorre o projeto **de ponta a ponta**, explicando cada etapa e mostrando os resultados.
Ele usa diretamente os módulos do pacote `demand_forecasting` (em `src/`), os mesmos que o `scripts/run_pipeline.py` usa.

**Fluxo:** dados → análise exploratória (EDA) → engenharia de atributos → modelos → avaliação → explicabilidade.

> 💡 Execute as células na ordem (menu *Run → Run All*, ou `Shift+Enter` célula a célula).
> A primeira execução baixa o dataset da UCI (~alguns MB) uma única vez.

## 0. Preparação do ambiente

Adicionamos a pasta `src/` ao caminho de importação para enxergar o pacote `demand_forecasting`.
Isso permite rodar o notebook sem instalar o pacote.

In [1]:
import sys
from pathlib import Path

# Caminho da raiz do projeto (um nível acima de notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from demand_forecasting import config, data, features, model, evaluate, explain

print('Projeto:', PROJECT_ROOT)
print('Pacote demand_forecasting importado com sucesso.')

Projeto: C:\Doutorado\Portfolio-GitHub\ml-demand-forecasting
Pacote demand_forecasting importado com sucesso.


## 1. Dados

`data.load_clean()` baixa o dataset **UCI Bike Sharing** (se ainda não existir localmente),
reconstrói um `timestamp` horário e remove colunas de identificador e de **vazamento**
(`casual` e `registered` somam o alvo `cnt`).

O alvo (`cnt`) é o total de aluguéis de bicicleta por hora.

In [2]:
df = data.load_clean()
print('Formato (linhas, colunas):', df.shape)
print('Período:', df['timestamp'].min(), '->', df['timestamp'].max())
df.head()

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)>

## 2. Análise exploratória (EDA)

Antes de modelar, olhamos a estrutura temporal da demanda. Dois padrões clássicos deste
problema são a **sazonalidade por hora do dia** e a diferença entre **dias úteis e fins de semana**.

In [ ]:
# Demanda média por hora do dia
by_hour = df.groupby(df['timestamp'].dt.hour)[config.TARGET].mean()

fig, ax = plt.subplots(figsize=(8, 4))
by_hour.plot(ax=ax, marker='o')
ax.set_xlabel('Hora do dia')
ax.set_ylabel('Aluguéis médios')
ax.set_title('Demanda média por hora do dia')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Distribuição do alvo
fig, ax = plt.subplots(figsize=(8, 4))
df[config.TARGET].plot(kind='hist', bins=50, ax=ax)
ax.set_xlabel('Aluguéis por hora (cnt)')
ax.set_title('Distribuição da demanda horária')
plt.show()

## 3. Engenharia de atributos

`features.build_features()` aplica três famílias de atributos:

1. **Calendário**: hora, dia da semana, mês, flag de fim de semana.
2. **Codificação cíclica** (sin/cos): ensina o modelo que a hora 23 é vizinha da hora 0.
3. **Lags e janelas móveis**: dão ao modelo a história recente da demanda (sem vazar o futuro).

As primeiras linhas, que ficam com `NaN` por causa dos lags, são descartadas.

In [ ]:
feat = features.build_features(df)
print('Antes:', df.shape, '-> Depois:', feat.shape)
# Mostra alguns dos atributos criados
novos = [c for c in feat.columns if c not in df.columns]
print('\nAtributos criados:')
print(novos)
feat[['timestamp', config.TARGET] + novos[:6]].head()

## 4. Divisão cronológica (treino/teste)

`model.chronological_split()` corta a série no tempo: o início vira **treino** e o final
vira **teste**. Nunca embaralhamos — isso evitaria o vazamento de informação do futuro.

In [ ]:
split = model.chronological_split(feat)
print('Treino:', split.X_train.shape[0], 'amostras')
print('Teste: ', split.X_test.shape[0], 'amostras')
print('Nº de atributos:', split.X_train.shape[1])

## 5. Treino, tuning e comparação de modelos

`model.build_models()` devolve três candidatos: um **baseline** (média), **Random Forest**
e **Gradient Boosting**. `model.tune_model()` faz busca de hiperparâmetros com validação
cruzada temporal (`TimeSeriesSplit`) para os modelos de árvore.

> ⏱️ Esta célula é a mais demorada (treina e tuna os modelos). Em um laptop típico leva alguns minutos.

In [ ]:
resultados = {}
modelos_treinados = {}

for nome, pipeline in model.build_models().items():
    print(f'Treinando: {nome} ...')
    melhor = model.tune_model(nome, pipeline, split)
    preds = melhor.predict(split.X_test)
    resultados[nome] = evaluate.regression_metrics(split.y_test, preds)
    modelos_treinados[nome] = melhor

ranking = evaluate.leaderboard(resultados)
print('\n=== Ranking (menor RMSE = melhor) ===')
ranking

## 6. Avaliação do melhor modelo

Pegamos o modelo de menor RMSE e inspecionamos os gráficos de diagnóstico:
**real vs. previsto** e a **distribuição dos resíduos**.

In [ ]:
melhor_nome = ranking.index[0]
melhor_modelo = modelos_treinados[melhor_nome]
preds = melhor_modelo.predict(split.X_test)
print('Melhor modelo:', melhor_nome)
print('Métricas:', resultados[melhor_nome])

# Real vs. previsto
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(split.y_test, preds, s=8, alpha=0.3)
lims = [min(split.y_test.min(), preds.min()), max(split.y_test.max(), preds.max())]
ax.plot(lims, lims, 'r--', linewidth=1)
ax.set_xlabel('Real')
ax.set_ylabel('Previsto')
ax.set_title(f'Real vs. Previsto — {melhor_nome}')
plt.show()

## 7. Explicabilidade

`explain.permutation_importances()` mede quanto o erro aumenta ao embaralhar cada atributo —
quanto maior, mais importante o atributo é para o modelo. Isso ajuda a **interpretar os drivers da demanda**.

In [ ]:
importancias = explain.permutation_importances(melhor_modelo, split.X_test, split.y_test)

top = importancias.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top['feature'], top['importance_mean'], xerr=top['importance_std'])
ax.set_xlabel('Importância por permutação (aumento no erro)')
ax.set_title('Top 15 atributos')
plt.show()

importancias.head(10)

## ✅ Conclusão

Percorremos todo o pipeline usando os módulos do pacote:

| Etapa | Módulo |
|---|---|
| Dados | `data.py` |
| Atributos | `features.py` |
| Divisão + modelos + tuning | `model.py` |
| Avaliação | `evaluate.py` |
| Explicabilidade | `explain.py` |

Para rodar tudo de uma vez (sem notebook), use no terminal:

```bash
python scripts/run_pipeline.py
```

Isso gera as métricas e os gráficos na pasta `outputs/`.